# 01 Clean data

Harmonize firm names and write cleaned tables to data/processed/.

Build data/processed/firm_crosswalk.csv mapping every raw manufacturer/applicant name
in PQR, WHO PQ FPP and WHO PQ API to a harmonized firm and corporate group.
Apply the firm crosswalk and write cleaned analysis tables to data/processed/.

In [ ]:
%run 00_utils.ipynb

import pandas as pd

## Firm crosswalk

In [ ]:
src = pd.concat([
    load_pqr().groupby("manufacturer").size().rename("n_obs").reset_index()
        .rename(columns={"manufacturer": "raw_name"}).assign(source="pqr"),
    load_pq_fpp().groupby("Applicant").size().rename("n_obs").reset_index()
        .rename(columns={"Applicant": "raw_name"}).assign(source="pq_fpp"),
    load_pq_api().groupby("Applicant").size().rename("n_obs").reset_index()
        .rename(columns={"Applicant": "raw_name"}).assign(source="pq_api"),
], ignore_index=True)
src["raw_name"] = src["raw_name"].str.strip()
src["key"] = src["raw_name"].map(norm_key)

rows = []
for r in src.itertuples():
    m = match(r.raw_name)
    if m is None:
        rows.append((None, None, None, "junk"))
    else:
        rows.append(m)
src[["firm", "group", "hq_country", "method"]] = pd.DataFrame(rows, index=src.index)

# Fallback: names with no rule share a firm if their normalized keys match.
# Display name = the most frequent raw spelling within the key.
fb = src[src.method == "normalized"]
disp = fb.sort_values("n_obs", ascending=False).groupby("key")["raw_name"].first()
src.loc[fb.index, "firm"] = fb["key"].map(disp)
src.loc[fb.index, "group"] = src.loc[fb.index, "firm"]
src.loc[fb.index, "method"] = fb["key"].map(fb.groupby("key").size()).map(
    lambda n: "normalized-merge" if n > 1 else "unchanged")

src["country_method"] = src["hq_country"].notna().map({True: "researcher-coded", False: None})
h = src.hq_country.isna() & src.raw_name.notna() & (src.method != "junk")
src.loc[h, "hq_country"] = src.loc[h, "raw_name"].map(country_heuristic)
src.loc[h & src.hq_country.notna(), "country_method"] = "name-heuristic"
src = src.drop(columns="key").sort_values(["firm", "source", "raw_name"])
out = ROOT / "data" / "processed" / "firm_crosswalk.csv"
src.to_csv(out, index=False)

print("raw names:", len(src), "| junk:", (src.method == "junk").sum())
for s in ["pqr", "pq_fpp", "pq_api"]:
    d = src[src.source == s]
    print(f"{s:7s} raw {d.raw_name.nunique():4d} -> firms {d.firm.nunique():4d} -> groups {d.group.nunique():4d}")
print("all sources: firms", src.firm.nunique(), "groups", src.group.nunique())
print(src.method.value_counts().to_string())
multi = src.groupby("firm")["raw_name"].nunique()
print("\nfirms with >1 raw spelling:", (multi > 1).sum())
print(src[src.method == "normalized-merge"][["raw_name", "firm", "source"]].to_string(index=False))

print("\nhq_country coverage by source (share of raw names):")
print(src[src.method != "junk"].groupby("source")["hq_country"].apply(lambda x: x.notna().mean()).round(2).to_string())

I matched every raw name to a firm and a corporate group, so spelling variants and mergers did not count as separate suppliers. The 326 raw names became 227 firms across the three sources. In PQR, 124 raw names became 110 firms and 104 groups. One name was a note and not a company, so I treated it as junk.

## Cleaned tables

In [ ]:
P = ROOT / "data" / "processed"
cw = pd.read_csv(P / "firm_crosswalk.csv")

def attach(df, col, source):
    m = cw[cw.source == source].set_index("raw_name")[["firm", "group", "hq_country"]]
    out = df.merge(m, left_on=df[col].str.strip(), right_index=True, how="left")
    return out.drop(columns="key_0", errors="ignore")

pqr = attach(load_pqr(), "manufacturer", "pqr")
pqr = pqr[pqr.firm.notna()]                       # drops 3 rows with an info-note instead of a name
pqr.to_csv(P / "pqr_medicines_clean.csv", index=False)
fpp = attach(load_pq_fpp(), "Applicant", "pq_fpp"); fpp.to_csv(P / "pq_fpp_clean.csv", index=False)
api = attach(load_pq_api(), "Applicant", "pq_api"); api.to_csv(P / "pq_api_clean.csv", index=False)

print("PQR rows", len(pqr), "| firms", pqr.firm.nunique(), "| groups", pqr.group.nunique())
spend = pqr.groupby("firm").cost_usd.sum()
print("spend share with hq_country:", round(spend[pqr.groupby("firm").hq_country.first().notna()].sum() / spend.sum(), 3))
print("\nspend share by HQ country:")
print((pqr.groupby("hq_country", dropna=False).cost_usd.sum() / pqr.cost_usd.sum()).sort_values(ascending=False).head(8).round(3).to_string())
in_pq = spend.index.isin(list(set(fpp.firm)))
print(f"\nPQR firms also WHO PQ FPP applicants: {in_pq.sum()} of {len(spend)} firms, "
      f"{spend[in_pq].sum() / spend.sum():.1%} of spend")
print("\ntop firms by spend:")
print((spend / spend.sum()).sort_values(ascending=False).head(8).round(3).to_string())

The clean PQR table had 74,689 medicine transactions after I dropped the 3 rows with a note in the manufacturer field. The HQ country was filled in for firms covering 97.8% of spend, so missing countries barely affect the spend shares. I coded HQ country by hand or from name patterns, so it still needs spot checks.

Firms based in India held 82% of spend, and firms based in China held 4.6%. The 46 PQR firms that are also WHO PQ applicants for finished products covered 96.2% of spend.